# Source: https://data.cityofnewyork.us/dataset/Hyperlocal-Temperature-Monitoring/qdq3-9eqn/about_data

In [1]:
# Import necessary libraries for data processing and nearest neighbor operations.
import pandas as pd  # For handling tabular data (CSV files).
import numpy as np  # For numerical operations and array manipulations.
import sklearn
from sklearn.neighbors import NearestNeighbors  # For KNN-based spatial matching.
import sys  # For accessing the Python version.

# Debug: Confirm that imports are successful.
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module.
print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"sklearn version: {sklearn.__version__}")

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
pandas version: 2.2.3
numpy version: 1.26.4
sklearn version: 1.2.2


In [2]:
# Define directory paths for Kaggle environment.
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets.
sub_dir = r"/kaggle/working/"  # Submission directory for output files.

# Define file paths for input datasets.
hyperlocal_file = f"{base_dir}/Hyperlocal_Temperature_Monitoring_20250311.csv"
train_file = f"{base_dir}/Training_data.csv"
valid_file = f"{base_dir}/Validation_data.csv"

# Define output file paths for merged datasets.
output_train_csv = f"{sub_dir}/Training_set_combined_with_TempMetrics.csv"
output_valid_csv = f"{sub_dir}/Validation_set_combined_with_TempMetrics.csv"

# Debug: Print the file paths to confirm they are set correctly.
print(f"Debug: Hyperlocal temperature file path: {hyperlocal_file}")
print(f"Debug: Training file path: {train_file}")
print(f"Debug: Validation file path: {valid_file}")
print(f"Debug: Output training CSV path: {output_train_csv}")
print(f"Debug: Output validation CSV path: {output_valid_csv}")

Debug: Hyperlocal temperature file path: /kaggle/input/eyds-base-dataset/Hyperlocal_Temperature_Monitoring_20250311.csv
Debug: Training file path: /kaggle/input/eyds-base-dataset/Training_data.csv
Debug: Validation file path: /kaggle/input/eyds-base-dataset/Validation_data.csv
Debug: Output training CSV path: /kaggle/working//Training_set_combined_with_TempMetrics.csv
Debug: Output validation CSV path: /kaggle/working//Validation_set_combined_with_TempMetrics.csv


In [3]:
# --- Step 1: Load the hyperlocal temperature dataset ---
# Source: https://data.cityofnewyork.us/dataset/Hyperlocal-Temperature-Monitoring/qdq3-9eqn/about_data
print("Loading hyperlocal temperature dataset...")
try:
    df_hyper = pd.read_csv(hyperlocal_file)
except FileNotFoundError:
    print(f"Error: Hyperlocal temperature file not found at {hyperlocal_file}")


# Debug: Print the shape and columns of the hyperlocal temperature dataset.
print(f"Debug: Hyperlocal DataFrame shape: {df_hyper.shape}")
print(f"Debug: Hyperlocal DataFrame columns: {df_hyper.columns.tolist()}")

# Convert AirTemp from Fahrenheit to Celsius.
df_hyper['AirTemp'] = (df_hyper['AirTemp'] - 32) * (5.0/9.0)

# Debug: Print a sample of the converted AirTemp values.
print(f"Debug: Sample AirTemp values (in Celsius): {df_hyper['AirTemp'].head().tolist()}")

# Extract month from the 'Day' column (assumed to be a date string like "2018-05-12").
# Note: The dataset does not have a separate 'Month' column, so we derive it.
df_hyper['Datetime'] = pd.to_datetime(df_hyper['Day'], errors='coerce')
df_hyper['Month'] = df_hyper['Datetime'].dt.month

# Debug: Print the unique months in the dataset.
print(f"Debug: Unique months in hyperlocal dataset: {df_hyper['Month'].unique()}")

# Filter for year 2018.
df_hyper = df_hyper[df_hyper['Year'] == 2018]

# Debug: Print the shape after filtering for year 2018.
print(f"Debug: Hyperlocal DataFrame shape after filtering for year 2018: {df_hyper.shape}")



Loading hyperlocal temperature dataset...
Debug: Hyperlocal DataFrame shape: (2097150, 10)
Debug: Hyperlocal DataFrame columns: ['Sensor.ID', 'AirTemp', 'Day', 'Hour', 'Latitude', 'Longitude', 'Year', 'Install.Type', 'Borough', 'ntacode']
Debug: Sample AirTemp values (in Celsius): [21.771666666666665, 21.246296294444445, 20.773703705555555, 20.146203705555557, 19.507777777777783]
Debug: Unique months in hyperlocal dataset: [ 6  7  8  9 10]
Debug: Hyperlocal DataFrame shape after filtering for year 2018: (1048575, 12)


In [4]:
# --- Step 2: Filter by month and time ---
# Define months of interest: June, July, August (skipping May since no data).
months_of_interest = [6, 7, 8]
# Filter for desired months and for Hour between 12 (noon) and 17 (5 PM) inclusive.
df_hyper_filtered = df_hyper[
    (df_hyper['Month'].isin(months_of_interest)) &
    (df_hyper['Hour'] >= 12) & (df_hyper['Hour'] <= 17)
]

# Debug: Print the shape after filtering by month and time.
print(f"Debug: Hyperlocal DataFrame shape after filtering by month and time: {df_hyper_filtered.shape}")

Debug: Hyperlocal DataFrame shape after filtering by month and time: (166608, 12)


In [5]:
# --- Step 3: Calculate temperature metrics per location and month ---
# Group by Latitude, Longitude, and Month and compute multiple metrics for AirTemp.
grouped = df_hyper_filtered.groupby(['Latitude', 'Longitude', 'Month'])['AirTemp'].agg(
    AvgTemp='mean',
    MinTemp='min',
    MaxTemp='max',
    StdTemp='std'
).reset_index()

# Debug: Print the shape of the grouped DataFrame and a sample.
print(f"Debug: Grouped DataFrame shape: {grouped.shape}")
print(f"Debug: Sample of grouped DataFrame:\n{grouped.head()}")

# --- Step 4: Calculate overall monthly average and UHI values ---
# Compute the overall monthly average temperature (across all locations).
overall_monthly = df_hyper_filtered.groupby('Month')['AirTemp'].mean().reset_index().rename(columns={'AirTemp': 'OverallAvgTemp'})

# Debug: Print the overall monthly averages.
print(f"Debug: Overall monthly averages:\n{overall_monthly}")

# Merge the overall monthly average back into the grouped data.
grouped = pd.merge(grouped, overall_monthly, on='Month', how='left')

# Compute the UHI value as the ratio of local AvgTemp to overall monthly average temperature.
grouped['UHI'] = grouped['AvgTemp'] / grouped['OverallAvgTemp']

# Debug: Print a sample of the grouped DataFrame with UHI values.
print(f"Debug: Sample of grouped DataFrame with UHI:\n{grouped.head()}")

Debug: Grouped DataFrame shape: (1068, 7)
Debug: Sample of grouped DataFrame:
    Latitude  Longitude  Month    AvgTemp    MinTemp    MaxTemp   StdTemp
0  40.646738 -73.951234      6  27.744553  19.301481  35.181667  4.043337
1  40.646738 -73.951234      7  29.250675  22.552963  37.544444  2.947602
2  40.646738 -73.951234      8  29.707823  21.576574  36.592593  4.009105
3  40.646877 -73.946154      6  27.385994  18.988519  34.303519  3.921132
4  40.646877 -73.946154      7  28.985300  22.249630  35.765370  2.684722
Debug: Overall monthly averages:
   Month  OverallAvgTemp
0      6       28.326396
1      7       29.767191
2      8       29.973970
Debug: Sample of grouped DataFrame with UHI:
    Latitude  Longitude  Month    AvgTemp    MinTemp    MaxTemp   StdTemp  \
0  40.646738 -73.951234      6  27.744553  19.301481  35.181667  4.043337   
1  40.646738 -73.951234      7  29.250675  22.552963  37.544444  2.947602   
2  40.646738 -73.951234      8  29.707823  21.576574  36.592593  4.00

In [6]:
# --- Step 5: Reshape the temperature metrics data to have one row per location ---
# Pivot the metrics so that each metric for each month becomes a separate column.
pivot = grouped.pivot_table(
    index=['Latitude', 'Longitude'],
    columns='Month',
    values=['AvgTemp', 'MinTemp', 'MaxTemp', 'StdTemp', 'UHI']
).reset_index()

# Flatten the multi-level columns.
month_mapping = {6: "June_2018", 7: "July_2018", 8: "Aug_2018"}
new_columns = {}
for col in pivot.columns:
    if isinstance(col, tuple):
        metric, month = col
        if month in month_mapping:
            new_columns[col] = f"{metric}_{month_mapping[month]}"
    else:
        new_columns[col] = col
pivot = pivot.rename(columns=new_columns)

# Debug: Print the shape and columns of the pivoted DataFrame.
print(f"Debug: Pivoted DataFrame shape: {pivot.shape}")
print(f"Debug: Pivoted DataFrame columns: {pivot.columns.tolist()}")

Debug: Pivoted DataFrame shape: (356, 17)
Debug: Pivoted DataFrame columns: [('Latitude', ''), ('Longitude', ''), ('AvgTemp', 6), ('AvgTemp', 7), ('AvgTemp', 8), ('MaxTemp', 6), ('MaxTemp', 7), ('MaxTemp', 8), ('MinTemp', 6), ('MinTemp', 7), ('MinTemp', 8), ('StdTemp', 6), ('StdTemp', 7), ('StdTemp', 8), ('UHI', 6), ('UHI', 7), ('UHI', 8)]


In [7]:
# --- Step 6: Load the training and validation datasets ---
print("Loading dataset...")
df_training = pd.read_csv(train_file)

# Debug: Print the shape and columns of the training dataset.
print(f"Debug: Training DataFrame shape: {df_training.shape}")
print(f"Debug: Training DataFrame columns: {df_training.columns.tolist()}")


df_validation = pd.read_csv(valid_file)

# Debug: Print the shape and columns of the validation dataset.
print(f"Debug: Validation DataFrame shape: {df_validation.shape}")
print(f"Debug: Validation DataFrame columns: {df_validation.columns.tolist()}")

Loading dataset...
Debug: Training DataFrame shape: (11229, 4)
Debug: Training DataFrame columns: ['Longitude', 'Latitude', 'datetime', 'UHI Index']
Debug: Validation DataFrame shape: (1040, 3)
Debug: Validation DataFrame columns: ['Longitude', 'Latitude', 'UHI Index']


In [8]:
# --- Step 7: Use KNN to join temperature metrics based on nearest lat/long using 1 neighbor ---
# Prepare the coordinates from the pivot temperature metrics dataset.
coords = pivot[['Latitude', 'Longitude']].values
knn = NearestNeighbors(n_neighbors=1, algorithm='auto')
knn.fit(coords)

# Debug: Confirm that the KNN model was fitted.
print("Debug: KNN model fitted successfully with 1 neighbor.")

def knn_merge(df, temp_df, knn_model):
    """
    Merge temperature metrics into a target DataFrame using KNN to find the nearest neighbor based on latitude and longitude.
    
    Parameters:
        df (pd.DataFrame): Target DataFrame with 'Latitude' and 'Longitude' columns.
        temp_df (pd.DataFrame): Temperature metrics DataFrame with metrics to merge.
        knn_model (NearestNeighbors): Fitted KNN model for finding nearest neighbors.
    
    Returns:
        pd.DataFrame: Target DataFrame with temperature metrics added from the nearest neighbor.
    """
    # Get coordinates for the target dataset.
    target_coords = df[['Latitude', 'Longitude']].values
    distances, indices = knn_model.kneighbors(target_coords)
    df = df.copy()
    
    # Get all temperature metric columns from temp_df except for Latitude and Longitude.
    temp_columns = [col for col in temp_df.columns if col not in ['Latitude', 'Longitude']]
    
    # For each temperature metric column, take the value from the nearest neighbor (since n_neighbors=1).
    for col in temp_columns:
        df[col] = [temp_df.iloc[idx[0]][col] for idx in indices]
    return df

# Merge temperature metrics using KNN matching for both datasets.
print("Merging temperature metrics for training data...")
df_training_merged = knn_merge(df_training, pivot, knn)

# Debug: Print the shape of the training data after merging.
print(f"Debug: Training DataFrame shape after merging: {df_training_merged.shape}")

print("Merging temperature metrics for validation data...")
df_validation_merged = knn_merge(df_validation, pivot, knn)

# Debug: Print the shape of the validation data after merging.
print(f"Debug: Validation DataFrame shape after merging: {df_validation_merged.shape}")

Debug: KNN model fitted successfully with 1 neighbor.
Merging temperature metrics for training data...
Debug: Training DataFrame shape after merging: (11229, 21)
Merging temperature metrics for validation data...
Debug: Validation DataFrame shape after merging: (1040, 20)


In [9]:
# --- Step 8: Save the merged datasets with new temperature metric columns ---
print("Saving merged datasets...")
df_training_merged.to_csv(output_train_csv, index=False)
df_validation_merged.to_csv(output_valid_csv, index=False)

# Debug: Print the final confirmation messages with file paths.
print(f"Debug: Merged training data saved to: {output_train_csv}")
print(f"Debug: Merged validation data saved to: {output_valid_csv}")

print("Temperature metrics (in Celsius) calculated using 1-nearest neighbor and datasets merged successfully.")

Saving merged datasets...
Debug: Merged training data saved to: /kaggle/working//Training_set_combined_with_TempMetrics.csv
Debug: Merged validation data saved to: /kaggle/working//Validation_set_combined_with_TempMetrics.csv
Temperature metrics (in Celsius) calculated using 1-nearest neighbor and datasets merged successfully.
